# Chapter 1: Prompt Chaining

EliteAI Academy

Build prompt chains for extraction, question answering, content, conversation, and code; compare native SDK and LCEL implementations.

## 0. Setup

Run `setup.ps1` from the course folder once, then select **Python (Agentic AI Course)** as the kernel.
The course loads your DeepSeek credentials from `.env`; restart the kernel after editing that file.
The install cell below is optional if setup is already complete. Run the remaining cells in order.

**Teaching check:** explain the pattern, run one example, inspect its output, then try an exercise.
Live model calls use API credits and may produce different answers. Stock prices, weather, CRM records,
email delivery, and approval workflows in these lessons are classroom simulations. Model-generated
research is illustrative and needs independent source verification before factual use.


In [ ]:
%pip install -r requirements-lock.txt --quiet

In [ ]:
from course_config import MODEL
import os
import json

import anthropic
client = anthropic.Anthropic(timeout=60.0, max_retries=2)

print("✅ DeepSeek client ready. Model:", MODEL)
from course_utils import parse_json_response


## 1. Helper Utility — `call_llm`

A thin wrapper so every example stays clean and readable.

In [ ]:
def call_llm(prompt: str, system: str = "You are a helpful AI assistant.") -> str:
    'Send a single prompt to the LLM and return the text response.'
    response = client.messages.create(thinking={"type": "disabled"}, 
        model=MODEL,
        max_tokens=1024,
        system=system,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.content[0].text

print(call_llm("Say hello in one sentence."))

## 2. Why Single Prompts Fail — Demonstration

We ask ONE prompt to do everything: summarise → identify trends → draft an email. Notice how quality degrades when the cognitive load is high.

In [ ]:
MARKET_REPORT = """
Global e-commerce sales reached $5.8 trillion in 2023, up 10% year-on-year.
AI-powered personalisation now influences 73% of consumer purchase decisions.
Sustainable products with ESG-related claims grew 28% over five years vs 20% for non-ESG.
Mobile commerce accounts for 60% of all online transactions globally.
Supply chain disruptions cost retailers an average of $184 million per incident in 2023.
"""

monolithic_prompt = f"""
You are a market analyst, trade analyst, AND expert documentation writer.

Given the following market research report:
{MARKET_REPORT}

Please do ALL of the following in one response:
1. Summarise the key findings in 3 bullet points.
2. Identify the top 3 emerging trends and list exact data points for each.
3. Draft a professional email to the marketing team covering all trends and data.
"""

single_result = call_llm(monolithic_prompt)
print("=== MONOLITHIC PROMPT OUTPUT ===")
print(single_result)
print("\n⚠️  Notice: mixed quality, hard to validate each section independently.")

## 3. Core Pattern — Three-Step Prompt Chain

We split the task above into three focused prompts, each with a dedicated role.

In [ ]:
step1_prompt = f"Summarise the key findings of the following market research report in 3 concise bullet points:\n\n{MARKET_REPORT}"

summary = call_llm(step1_prompt, system="You are a senior Market Analyst. Be precise and concise.")
print("[STEP 1] Summary\n", summary)
print("-" * 60)

In [ ]:
step2_prompt = f"""
Using the summary below, identify the top 3 emerging market trends.
Return ONLY a valid JSON object — no markdown, no explanation.

Required format:
{{
  "trends": [
    {{"trend_name": "...", "supporting_data": "..."}},
    {{"trend_name": "...", "supporting_data": "..."}},
    {{"trend_name": "...", "supporting_data": "..."}}
  ]
}}

Summary:
{summary}
"""

trends_raw = call_llm(step2_prompt, system="You are a Trade Analyst. Always respond with valid JSON only.")
trends_data = parse_json_response(trends_raw)

print("[STEP 2] Structured Trends (parsed JSON)")
print(json.dumps(trends_data, indent=2))
print("-" * 60)

In [ ]:
trends_text = "\n".join(
    f"- {t['trend_name']}: {t['supporting_data']}" for t in trends_data["trends"]
)

step3_prompt = f"""
Draft a concise, professional email to the marketing team covering the following trends and data.
Include a subject line.

Trends:
{trends_text}
"""

email_draft = call_llm(step3_prompt, system="You are an Expert Documentation Writer. Tone: professional.")
print("[STEP 3] Email Draft")
print(email_draft)

### Key Observation

Focused steps pass structured data forward; validate JSON before using it.

## 4. Use Case 1 — Information Processing Pipeline

Extract → Summarise → Identify entities → Report.

In [ ]:
raw_article = """
On 15 March 2024, OpenAI CEO Sam Altman met with EU Commissioner Thierry Breton in Brussels.
They discussed the EU AI Act, signed into law in February 2024, which imposes strict 
obligations on high-risk AI systems. OpenAI's GPT-4 model has been flagged as a General 
Purpose AI (GPAI) model requiring additional transparency obligations. The meeting took place 
at the European Commission headquarters. Fines for non-compliance can reach €30 million 
or 6% of global turnover.
"""

s1 = call_llm(f"Summarise this article in 2 sentences:\n\n{raw_article}")

s2_prompt = f"""
Extract named entities from the text below.
Return ONLY valid JSON with keys: people, organisations, locations, dates, monetary_values.

Text: {raw_article}
"""
s2_raw = call_llm(s2_prompt)
entities = parse_json_response(s2_raw)

s3_prompt = f"""
Write a brief analyst report using the summary and entities below.

Summary: {s1}
Entities: {json.dumps(entities)}
"""
report = call_llm(s3_prompt)

print("SUMMARY:", s1)
print("\nENTITIES:", json.dumps(entities, indent=2))
print("\nFINAL REPORT:\n", report)

## 5. Use Case 2 — Complex Multi-Hop Question Answering

Decompose a hard question → answer each sub-question → synthesise.

In [ ]:
complex_question = "What were the main causes of the 1929 stock market crash, and how did government policy respond?"

decompose_prompt = f"""
Break the following complex question into 2–3 focused sub-questions.
Return ONLY a JSON array of strings.

Question: {complex_question}
"""
sub_questions = parse_json_response(call_llm(decompose_prompt))
print("Sub-questions:", sub_questions)

answers = {}
for sq in sub_questions:
    answers[sq] = call_llm(f"Answer concisely (3–4 sentences): {sq}")
    print(f"\n[Q]: {sq}\n[A]: {answers[sq]}")

synthesis_prompt = f"""
Synthesise the following question-answer pairs into a single, coherent paragraph response 
to the original question: '{complex_question}'

Q&A pairs:
{json.dumps(answers, indent=2)}
"""
final_answer = call_llm(synthesis_prompt)
print("\n=== FINAL SYNTHESISED ANSWER ===")
print(final_answer)

## 6. Use Case 3 — Data Extraction & Transformation (Invoice OCR)

Extract fields → Validate → Re-prompt for missing fields → Output clean JSON

In [ ]:
invoice_text = """
INVOICE #INV-2024-0892
Vendor: DataTech Supplies Ltd.
Date: Fourteenth of June twenty twenty-four
Items:
  - 3x USB-C Hub @ $49.99 each
  - 1x Laptop Stand @ $89.00
Subtotal: one hundred and thirty-nine dollars and ninety-seven cents
Tax (8%): eleven dollars and twenty cents
TOTAL DUE: one hundred and fifty-one dollars and seventeen cents
"""

REQUIRED_FIELDS = ["invoice_number", "vendor", "date", "subtotal_usd", "tax_usd", "total_usd"]

extract_prompt = f"""
Extract and normalise the following fields from this invoice text.
Convert all written numbers to numeric values (e.g., "one hundred" → 100.00).
Convert dates to ISO 8601 (YYYY-MM-DD).
Return ONLY valid JSON.

Required fields: {REQUIRED_FIELDS}

Invoice:
{invoice_text}
"""
extracted_raw = call_llm(extract_prompt)
extracted = parse_json_response(extracted_raw)
print("[STEP 1] Extracted:", json.dumps(extracted, indent=2))

missing = [f for f in REQUIRED_FIELDS if not extracted.get(f)]
print("\n[STEP 2] Missing fields:", missing if missing else "None — all fields present ✅")

if missing:
    retry_prompt = f"""
    The following fields were not extracted from the invoice: {missing}.
    Please extract ONLY these fields and return a JSON object.
    
    Invoice:
    {invoice_text}
    """
    retry_raw = call_llm(retry_prompt)
    retry_data = parse_json_response(retry_raw)
    extracted.update(retry_data)
    print("\n[STEP 3] After retry:", json.dumps(extracted, indent=2))

computed_total = round(extracted["subtotal_usd"] + extracted["tax_usd"], 2)
match = computed_total == extracted["total_usd"]
print(f"\n[STEP 4] Computed total: ${computed_total} | Extracted total: ${extracted['total_usd']} | Match: {match}")
print("\n✅ Final validated invoice data:", json.dumps(extracted, indent=2))

## 7. Use Case 4 — Content Generation Workflow

Ideas → Select → Outline → Draft sections → Final review

In [ ]:
user_interest = "Agentic AI systems in enterprise settings"

ideas_prompt = f"""
Generate 5 specific blog post topic ideas based on the interest: '{user_interest}'.
Return ONLY a JSON array of 5 title strings.
"""
ideas = parse_json_response(call_llm(ideas_prompt))
print("[STEP 1] Topic Ideas:")
for i, idea in enumerate(ideas, 1):
    print(f"  {i}. {idea}")

selected_topic = ideas[0]
print(f"\n[STEP 2] Selected topic: '{selected_topic}'")

outline_prompt = f"""
Create a 3-section blog post outline for the topic: '{selected_topic}'.
Return ONLY a JSON array of section title strings.
"""
outline = parse_json_response(call_llm(outline_prompt))
print("\n[STEP 3] Outline:", outline)

sections = []
previous_content = ""
for section_title in outline:
    context = f"Previous content for continuity:\n{previous_content}\n\n" if previous_content else ""
    draft_prompt = f"""
    {context}Write a 3–4 sentence draft for the blog section titled: '{section_title}'.
    Topic: {selected_topic}
    """
    section_text = call_llm(draft_prompt)
    sections.append({"title": section_title, "content": section_text})
    previous_content = section_text
    print(f"\n[STEP 4] Section '{section_title}':\n{section_text}")

full_draft = "\n\n".join(f"## {s['title']}\n{s['content']}" for s in sections)
review_prompt = f"""
Review and lightly polish the following blog post draft for coherence, tone, and flow.
Keep the same structure. Return the improved version only.

{full_draft}
"""
final_post = call_llm(review_prompt)
print("\n=== FINAL POLISHED BLOG POST ===")
print(final_post)

## 8. Use Case 5 — Conversational Agent with State

Each turn builds on accumulated conversation history.

In [ ]:
def stateful_agent_turn(conversation_history: list, user_input: str) -> tuple:
    'Process one conversational turn.'

    conversation_history.append({"role": "user", "content": user_input})

    response = client.messages.create(thinking={"type": "disabled"}, 
        model=MODEL,
        max_tokens=512,
        system="You are a helpful travel planning assistant. Remember all details the user shares.",
        messages=conversation_history
    )
    assistant_reply = response.content[0].text

    conversation_history.append({"role": "assistant", "content": assistant_reply})

    return conversation_history, assistant_reply

history = []
turns = [
    "Hi! I want to plan a trip to Japan.",
    "I'll be travelling with my partner and we have about 10 days in mid-April.",
    "Based on what I told you, what are the top 3 things we should not miss?"
]

for turn in turns:
    history, reply = stateful_agent_turn(history, turn)
    print(f"\n👤 User: {turn}")
    print(f"🤖 Agent: {reply}")
    print("-" * 60)

print(f"\nTotal conversation turns stored: {len(history)}")

## 9. Use Case 6 — Code Generation & Refinement Pipeline

In [ ]:
user_request = "Write a Python function that reads a CSV file and returns rows where a given column value exceeds a threshold."

plan = call_llm(
    f"Write pseudocode (bullet points) for this request: {user_request}",
    system="You are a senior software architect. Be precise and structured."
)
print("[STEP 1] Plan / Pseudocode:\n", plan)

code_prompt = f"""
Based on this plan, write clean Python code with type hints and a docstring.
Return ONLY the code block, no explanation.

Plan:
{plan}
"""
initial_code = call_llm(code_prompt, system="You are an expert Python developer.")
print("\n[STEP 2] Initial Code:\n", initial_code)

review_prompt = f"""
Review the following Python code for:
- Potential bugs or edge cases
- Missing error handling
- PEP 8 style issues

List issues as JSON array of strings. If no issues, return [].

Code:
{initial_code}
"""
issues_raw = call_llm(review_prompt, system="You are a Python code reviewer.")
issues = parse_json_response(issues_raw)
print("\n[STEP 3] Review Issues:", issues)

if issues:
    refine_prompt = f"""
    Fix the following issues in the Python code and return ONLY the improved code.
    
    Issues: {issues}
    
    Original code:
    {initial_code}
    """
    refined_code = call_llm(refine_prompt, system="You are an expert Python developer.")
    print("\n[STEP 4] Refined Code:\n", refined_code)
else:
    refined_code = initial_code
    print("\n[STEP 4] No issues found — original code is final.")

tests_prompt = f"""
Write 3 pytest test cases for the following function.
Return ONLY the test code.

Function:
{refined_code}
"""
tests = call_llm(tests_prompt)
print("\n[STEP 5] Test Cases:\n", tests)

## 10. LangChain / LCEL Implementation

The same two-step extraction → JSON transformation chain from the book, using LangChain Expression Language.

In [ ]:
from langchain_anthropic import ChatAnthropic
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

lc_llm = ChatAnthropic(model_kwargs={"thinking": {"type": "disabled"}}, model=MODEL, temperature=0)

prompt_extract = ChatPromptTemplate.from_template(
    "Extract the technical specifications from the following text:\n\n{text_input}"
)

prompt_transform = ChatPromptTemplate.from_template(
    """Transform the following specifications into a JSON object 
    with 'cpu', 'memory', and 'storage' as keys.
    Return ONLY valid JSON, no markdown.
    
    Specifications:\n\n{specifications}"""
)

extraction_chain = prompt_extract | lc_llm | StrOutputParser()

full_chain = (
    {"specifications": extraction_chain}
    | prompt_transform
    | lc_llm
    | StrOutputParser()
)

input_text = "The new laptop model features a 3.5 GHz octa-core processor, 16GB of RAM, and a 1TB NVMe SSD."
final_result = full_chain.invoke({"text_input": input_text})

print("--- LangChain LCEL Chain Output ---")
print(final_result)
parsed = parse_json_response(final_result)
print("\nParsed Python dict:", parsed)

## 11. Summary — Pattern Rules of Thumb

In [ ]:
summary_table = """
╔══════════════════════════════════╦═════════════════════════════════════════════╗
║ When to USE Prompt Chaining      ║ Key Design Rules                            ║
╠══════════════════════════════════╬═════════════════════════════════════════════╣
║ Task has multiple distinct stages║ Use JSON/structured output between steps    ║
║ Single prompt produces mixed     ║ Assign a specific role per step             ║
║ quality output                   ║ Validate output before passing it forward   ║
║ Steps depend on previous results ║ Insert deterministic logic between LLM calls║
║ External tool calls needed       ║ Use conditional re-prompts for missing data ║
║ Multi-step reasoning required    ║ Combine parallel + sequential as needed     ║
╚══════════════════════════════════╩═════════════════════════════════════════════╝
"""
print(summary_table)

## 12. Exercises

- **Exercise 1 — Extend Use Case 1:** Add a Step 4 that queries Wikipedia (or a mock knowledge base dict) using extracted entities and appends the result to the report.

- **Exercise 2 — Parallel + Sequential:** Given 3 news articles, summarise each concurrently (use `concurrent.futures.ThreadPoolExecutor`), then chain a synthesis step.

- **Exercise 3 — Multimodal Chain:** Upload an image of a product label. Chain: (1) extract text from image → (2) identify ingredients → (3) flag any allergens → (4) generate a consumer safety summary.

- **Exercise 4 — Error Recovery:** Modify the invoice extractor to retry up to 3 times if JSON parsing fails, logging each attempt.